# Praktikum 03 — Pengelolaan Data dan Privasi
**Dataset:** Penjualan UMKM 2024  
**Nama:** Aqila Diliansyah Putra  
**NIM:** 0110225025

Notebook ini menggunakan dataset transaksi UMKM yang diunggah. Jalankan sel dari atas ke bawah. Sebelum membagikan dataset, periksa apakah ada informasi yang bisa mengungkap identitas pelanggan atau pelaku usaha.

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from google.colab import files

uploaded = files.upload()
FILE = next(iter(uploaded))
df = pd.read_csv(FILE)

print("Nama file:", FILE)
print("Ukuran data:", df.shape)
display(df.head())

## A. Pemeriksaan awal dataset

In [ ]:
# A1 — Struktur dan tipe data
print("Ukuran (baris, kolom):", df.shape)
print("\nTipe data:")
display(df.dtypes.rename("tipe_data").to_frame())
print("\nNama kolom:")
print(df.columns.tolist())

In [ ]:
# A2 — Ringkasan kolom numerik dan kategorikal
display(df.describe(include="all").T)

### A3 — Pertanyaan analisis
Contoh pertanyaan yang bisa dijawab dari kolom dataset:
1. Apakah jumlah barang berkaitan dengan total pembayaran?
2. Produk atau kategori apa yang memiliki total penjualan tertinggi?
3. Apakah nilai transaksi berbeda menurut metode pembayaran?

Catatan: data ini berisi catatan transaksi; tidak bisa dipakai untuk menyimpulkan perubahan omzet suatu usaha dari waktu ke waktu tanpa informasi tambahan yang sesuai.

In [ ]:
# A4 — Temukan kolom yang berkaitan dengan transaksi atau pembayaran
[c for c in df.columns if any(k in c.lower() for k in ["total", "harga", "jumlah", "bayar"])]

## B. Skala pengukuran dan tipe tanggal

In [ ]:
# B1 — Tentukan skala pengukuran berdasarkan makna kolom
skala = {
    "id_transaksi": "Nominal (pengenal transaksi)",
    "tanggal": "Interval (jarak waktu bermakna)",
    "kota": "Nominal",
    "kategori": "Nominal",
    "produk": "Nominal",
    "jumlah": "Rasio",
    "harga_satuan": "Rasio",
    "diskon_persen": "Rasio",
    "metode_bayar": "Nominal",
    "rating_pelanggan": "Ordinal (rating berurutan)",
    "total_bayar": "Rasio"
}
tabel_skala = pd.DataFrame({
    "nama_kolom": df.columns,
    "skala_pengukuran": [skala.get(c, "Perlu ditentukan sesuai makna kolom") for c in df.columns]
})
display(tabel_skala)

In [ ]:
# B2 — Konversi tanggal dan pemeriksaan rentang
if "tanggal" in df.columns:
    df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")
    print("Tanggal paling awal:", df["tanggal"].min())
    print("Tanggal paling akhir:", df["tanggal"].max())
    print("Tanggal gagal dikonversi:", df["tanggal"].isna().sum())

## C. Kamus data dan pemeriksaan nilai kosong

In [ ]:
# C1 — Bangkitkan kerangka kamus data dari dataset
kamus = pd.DataFrame({
    "nama_kolom": df.columns,
    "tipe_data": [str(df[c].dtype) for c in df.columns],
    "jumlah_nilai_unik": [df[c].nunique(dropna=True) for c in df.columns],
    "jumlah_kosong": [df[c].isna().sum() for c in df.columns],
    "contoh_nilai": [df[c].dropna().iloc[0] if df[c].notna().any() else np.nan for c in df.columns]
})
display(kamus)

In [ ]:
# C2 — Daftar kolom yang mengandung nilai kosong
kamus[kamus["jumlah_kosong"] > 0]

### C3 — Catatan kamus data
- `id_transaksi`: kode unik untuk membedakan transaksi.
- `tanggal`: tanggal transaksi dilakukan.
- `kota`: kota transaksi/pelanggan tercatat.
- `kategori` dan `produk`: pengelompokan barang yang dijual.
- `jumlah`: banyaknya unit dalam transaksi.
- `harga_satuan`: harga satu unit sebelum perhitungan total.
- `diskon_persen`: persentase diskon transaksi.
- `metode_bayar`: cara pembayaran.
- `rating_pelanggan`: penilaian pelanggan.
- `total_bayar`: jumlah pembayaran akhir yang tercatat.

Deskripsi ini berdasarkan nama kolom dan contoh data; konfirmasikan dengan pemilik dataset jika definisi bisnisnya berbeda.

## D. Identifikasi data pribadi dan strategi penanganan

In [ ]:
# D1 — Penandaan kolom secara awal
# Ini merupakan evaluasi awal, bukan kepastian hukum atas klasifikasi data.
PENANDAAN = {
    "id_transaksi": ("pengenal transaksi", "hash"),
    "tanggal": ("quasi-identifier", "generalisasi"),
    "kota": ("quasi-identifier", "generalisasi"),
    "kategori": ("data transaksi", "pertahankan"),
    "produk": ("data transaksi", "pertahankan"),
    "jumlah": ("data transaksi", "pertahankan"),
    "harga_satuan": ("data transaksi", "pertahankan"),
    "diskon_persen": ("data transaksi", "pertahankan"),
    "metode_bayar": ("data transaksi", "pertahankan"),
    "rating_pelanggan": ("data transaksi", "pertahankan"),
    "total_bayar": ("data transaksi", "pertahankan")
}
tanda = pd.DataFrame([
    {"nama_kolom": c,
     "kategori": PENANDAAN.get(c, ("perlu ditinjau", "perlu ditinjau"))[0],
     "tindakan": PENANDAAN.get(c, ("perlu ditinjau", "perlu ditinjau"))[1]}
    for c in df.columns
])
display(tanda)

### D2 — Alasan penandaan
`id_transaksi` diperlakukan sebagai pengenal unik dan dapat diganti dengan token hash. `tanggal` dan `kota` dapat menjadi quasi-identifier jika digabungkan dengan informasi lain. Kolom transaksi lain dipertahankan untuk analisis, tetapi perlu diperiksa kembali berdasarkan konteks, izin, dan risiko publikasi.

## E. Membuat salinan data yang lebih aman

In [ ]:
# E1 — Terapkan penghapusan kolom yang memang ditandai 'hapus'
kolom_hapus = [c for c in df.columns if PENANDAAN.get(c, ("", "pertahankan"))[1] == "hapus"]
aman = df.drop(columns=kolom_hapus).copy()
print("Kolom yang dihapus:", kolom_hapus)
print("Ukuran data aman sementara:", aman.shape)

In [ ]:
# E2 — Hash pengenal transaksi
def samarkan(nilai, garam):
    if pd.isna(nilai):
        return np.nan
    return hashlib.sha256((garam + str(nilai)).encode("utf-8")).hexdigest()[:12]

# Ganti nilai ini sebelum dipakai; jangan simpan garam privat ke repositori publik.
GARAM = "ubah-menjadi-rahasia-lokal"
kolom_hash = [c for c in df.columns if PENANDAAN.get(c, ("", "pertahankan"))[1] == "hash"]

for c in kolom_hash:
    aman[c] = df[c].apply(lambda x: samarkan(x, GARAM))

print("Kolom yang di-hash:", kolom_hash)
if kolom_hash:
    display(aman[kolom_hash].head())

In [ ]:
# E3 — Uji konsistensi hasil hash
if kolom_hash and len(df) >= 2:
    c = kolom_hash[0]
    h1 = samarkan(df[c].iloc[0], GARAM)
    h1_ulang = samarkan(df[c].iloc[0], GARAM)
    h2 = samarkan(df[c].iloc[1], GARAM)
    print("Nilai sama menghasilkan hash sama:", h1 == h1_ulang)
    print("Dua nilai berbeda menghasilkan hash berbeda:", h1 != h2)
else:
    print("Perlu kolom hash dan minimal dua baris untuk uji.")

### E4 — Generalisasi tanggal dan kota
Tanggal dapat diubah menjadi bulan atau tahun, sedangkan kota dapat digabung ke wilayah yang lebih luas bila informasi wilayah tersedia. Contoh berikut menghasilkan kolom bulan dan tidak mempertahankan tanggal persis.

In [ ]:
if "tanggal" in aman.columns:
    aman["bulan_transaksi"] = pd.to_datetime(aman["tanggal"], errors="coerce").dt.to_period("M").astype(str)
    aman = aman.drop(columns=["tanggal"])
if "kota" in aman.columns:
    aman["kota"] = aman["kota"].astype("string").str.strip().str.title()

display(aman.head())

## F. Pemeriksaan risiko kelompok kecil (k-anonymity sederhana)

In [ ]:
# F1 — Fungsi menghitung ukuran kelompok terkecil pada kombinasi quasi-identifier
def cek_k(data, kolom):
    tersedia = [c for c in kolom if c in data.columns]
    if not tersedia:
        return pd.Series({"k": np.nan, "jumlah_grup": 0, "grup_satu_record": 0})
    ukuran_grup = data.groupby(tersedia, dropna=False, observed=True).size()
    return pd.Series({
        "k": int(ukuran_grup.min()) if len(ukuran_grup) else np.nan,
        "jumlah_grup": len(ukuran_grup),
        "grup_satu_record": int((ukuran_grup == 1).sum())
    })

In [ ]:
# F2 — Periksa kombinasi wilayah dan kategori yang tersedia
kolom_qi = [c for c in ["kota", "bulan_transaksi", "metode_bayar"] if c in aman.columns]
print("Kolom yang diperiksa:", kolom_qi)
display(cek_k(aman, kolom_qi))

In [ ]:
# F3 — Tabel agregat dan identifikasi sel berjumlah kecil
if "kota" in aman.columns and "kategori" in aman.columns:
    tabel_silang = pd.crosstab(aman["kota"], aman["kategori"])
    display(tabel_silang)
    sel_kecil = ((tabel_silang > 0) & (tabel_silang < 5)).sum().sum()
    print("Jumlah sel terisi dengan jumlah kurang dari 5:", int(sel_kecil))
else:
    print("Kolom kota/kategori tidak ditemukan.")

### F4 — Interpretasi
Nilai `k` yang kecil menunjukkan kombinasi atribut yang lebih jarang muncul sehingga berpotensi memudahkan pencocokan dengan data lain. Pemeriksaan ini hanya ukuran risiko sederhana; bukan jaminan anonimitas. Untuk publikasi, pertimbangkan menyamarkan atau menggabungkan kelompok kecil.

## G. Analisis ringkas dataset proyek

In [ ]:
# G1 — Ringkasan nilai transaksi menurut kategori
if {"kategori", "total_bayar"}.issubset(df.columns):
    ringkasan_kategori = df.groupby("kategori", dropna=False)["total_bayar"].agg(
        jumlah_transaksi="count", total_penjualan="sum", rata_rata_transaksi="mean"
    ).sort_values("total_penjualan", ascending=False)
    display(ringkasan_kategori)

In [ ]:
# G2 — Ringkasan menurut metode pembayaran
if {"metode_bayar", "total_bayar"}.issubset(df.columns):
    display(df.groupby("metode_bayar", dropna=False)["total_bayar"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False))

In [ ]:
# G3 — Periksa hubungan numerik antarvariabel
kolom_numerik = df.select_dtypes(include="number").columns
display(df[kolom_numerik].corr(numeric_only=True))

### G4 — Pemeriksaan kualitas data
Periksa duplikasi, nilai kosong, dan rentang angka sebelum menarik kesimpulan. Hasil ini bersifat deskriptif dan tidak membuktikan hubungan sebab-akibat.

In [ ]:
print("Ukuran data:", df.shape)
print("\nJumlah nilai kosong per kolom:")
display(df.isna().sum().rename("nilai_kosong").to_frame())
print("\nJumlah baris duplikat:", int(df.duplicated().sum()))
for c in df.select_dtypes(include="number").columns:
    print(f"{c}: min={df[c].min()}, max={df[c].max()}")

## G5 — Checklist sebelum Git Push
- [ ] Nama file dan sumber dataset sudah dicatat.
- [ ] Hak penggunaan/lisensi dataset sudah diperiksa.
- [ ] Pengenal unik sudah ditangani sesuai kebutuhan.
- [ ] Informasi sensitif dan quasi-identifier sudah ditinjau.
- [ ] Nilai kosong dan data duplikat sudah diperiksa.
- [ ] Definisi kolom sudah dicatat dalam kamus data.
- [ ] Tidak ada kata sandi, token, atau garam hash rahasia di repositori.
- [ ] Hasil analisis tidak ditafsirkan sebagai bukti sebab-akibat.